In [ ]:
import pandas as pd
import sqlite3

customers = pd.DataFrame({
    "customer_id": [101,102,103,104,105,106,107,108,109,110],
    "city": ["Nairobi","nairobi","Mombasa","Kisumu","Nairobi","Nakuru","Mombasa","Nairobi","Eldoret","Kisumu"],
    "signup_date": ["2026-01-14","2026-02-03","2026-06-20","2026-03-02","2026-03-15","2026-04-01","2026-04-18","2026-05-09","2026-06-21","2026-07-05"],
    "channel": ["Organic","Referral","Paid Social","Organic","Paid Social","Referral","Organic","Paid Social","Organic","Paid Social"],
})

orders = pd.DataFrame({
    "order_id": [1001,1002,1003,1004,1005,1006,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020],
    "customer_id": [101,102,103,104,101,105,105,106,102,107,101,108,103,104,105,102,106,101,108,107,999],
    "order_date": ["2026-06-03","2026-06-05","2026-06-11","2026-06-18","2026-06-25","2026-07-02","2026-07-02","2026-07-09","2026-07-14",
                    "2026-07-21","2026-07-28","2026-08-04","2026-08-09","2026-08-15","2026-08-19","2026-08-27","2026-09-03","2026-09-08",
                    "2026-09-14","2026-09-22","2026-09-26"],
    "status": ["completed","completed","cancelled","completed","completed","completed","completed",
                "refunded","completed","completed","completed","completed","completed","completed",
                "cancelled","completed","completed","completed","refunded","completed","completed"],
    "amount_kes": [2400,1850,3200,2100,2650,1500,1500,2900,3100,None,2200,1750,2800,185000,
                   1600,2450,3300,2750,2000,2350,1900],
    "payment_method": ["M-Pesa","M-Pesa","Card","M-Pesa","Card","M-Pesa","M-Pesa","Card","M-Pesa","M-Pesa",
                       "M-Pesa","Card","M-Pesa","M-Pesa","M-Pesa","M-Pesa","Card","M-Pesa","M-Pesa","Card","M-Pesa"],
})

# Load into SQLite so you can write real SQL
conn = sqlite3.connect(":memory:")
customers.to_sql("customers", conn, index=False)
orders.to_sql("orders", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

q("SELECT * FROM orders LIMIT 5")

customers.loc[customers["customer_id"] == 105, "city"] = "Nairobi "
print(customers["city"].apply(repr).value_counts())   # should now show 'Nairobi ' once

customers.to_sql("customers", conn, index=False, if_exists="replace")
# 1. Duplicate rows / duplicate order IDs
print(orders[orders.duplicated(keep=False)])
print(orders["order_id"].duplicated().sum())

# 2. Missing values
print(orders.isna().sum())

# 3. Orders with no matching customer (orphans)
print(orders[~orders["customer_id"].isin(customers["customer_id"])])

# 4. Messy text (case, trailing spaces)
print(customers["city"].apply(repr).value_counts())

# 5. Outliers
print(orders["amount_kes"].describe())
print(orders.nlargest(3, "amount_kes"))

# 6. Order placed before the customer signed up
m = orders.merge(customers, on="customer_id")
print(m[m["order_date"] < m["signup_date"]])

# 7. Status values
print(orders["status"].value_counts())

# Cleaned copies (raw tables stay untouched)
customers_clean = customers.copy()
customers_clean["city"] = customers_clean["city"].str.strip().str.title()

orders_clean = orders.drop_duplicates().copy()                    # removes the duplicate 1006
orders_clean["is_orphan"] = (~orders_clean["customer_id"].isin(customers["customer_id"])).astype(int)
orders_clean["amount_missing"] = orders_clean["amount_kes"].isna().astype(int)
orders_clean["amount_outlier"] = (orders_clean["amount_kes"] > 50000).astype(int)   # flag, not delete

customers_clean.to_sql("customers_clean", conn, index=False, if_exists="replace")
orders_clean.to_sql("orders_clean", conn, index=False, if_exists="replace")

# Sanity checks
print(len(orders_clean))
print(customers_clean["city"].value_counts())
print(orders_clean[["is_orphan","amount_missing","amount_outlier"]].sum())


# Raw table (shows the problem)
q("""
SELECT status, COUNT(*) AS orders
FROM orders
GROUP BY status
ORDER BY orders DESC
""")

# Clean table (the real answer)
q("""
SELECT status, COUNT(*) AS orders
FROM orders_clean
GROUP BY status
ORDER BY orders DESC
""")

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

# Version A: all completed revenue (includes the 185,000 order)
print(q("""
SELECT COALESCE(c.city, 'Unknown') AS city,
       COUNT(*)                    AS orders,
       SUM(o.amount_kes)           AS revenue_kes
FROM orders_clean o
LEFT JOIN customers_clean c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY COALESCE(c.city, 'Unknown')
ORDER BY revenue_kes DESC
"""))

# Version B: excluding the suspected outlier
print(q("""
SELECT COALESCE(c.city, 'Unknown') AS city,
       COUNT(*)                    AS orders,
       SUM(o.amount_kes)           AS revenue_kes
FROM orders_clean o
LEFT JOIN customers_clean c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
  AND o.amount_outlier = 0
GROUP BY COALESCE(c.city, 'Unknown')
ORDER BY revenue_kes DESC
"""))

print(q("""
SELECT c.customer_id, c.city, c.signup_date, c.channel
FROM customers_clean c
LEFT JOIN orders_clean o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
"""))

# Step 1: completed orders per customer (to see the workings)
print(q("""
SELECT c.customer_id,
       SUM(CASE WHEN o.status = 'completed' THEN 1 ELSE 0 END) AS completed_orders
FROM customers_clean c
LEFT JOIN orders_clean o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY c.customer_id
"""))

# Step 2: the repeat purchase rate
print(q("""
WITH per_customer AS (
  SELECT c.customer_id,
         SUM(CASE WHEN o.status = 'completed' THEN 1 ELSE 0 END) AS completed_orders
  FROM customers_clean c
  LEFT JOIN orders_clean o ON c.customer_id = o.customer_id
  GROUP BY c.customer_id
)
SELECT COUNT(*)                                   AS all_customers,
       SUM(completed_orders >= 1)                 AS buyers,
       SUM(completed_orders >= 2)                 AS repeat_buyers,
       ROUND(100.0 * SUM(completed_orders >= 2) / SUM(completed_orders >= 1), 1) AS repeat_rate_of_buyers,
       ROUND(100.0 * SUM(completed_orders >= 2) / COUNT(*), 1)                   AS repeat_rate_of_all
FROM per_customer
"""))

# Version A: all completed revenue
print(q("""
WITH monthly AS (
  SELECT strftime('%Y-%m', order_date) AS month,
         SUM(amount_kes)               AS revenue_kes
  FROM orders_clean
  WHERE status = 'completed'
  GROUP BY strftime('%Y-%m', order_date)
)
SELECT month,
       revenue_kes,
       LAG(revenue_kes) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0 * (revenue_kes - LAG(revenue_kes) OVER (ORDER BY month))
             / LAG(revenue_kes) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month
"""))

# Version B: excluding the suspected outlier (order 1013)
print(q("""
WITH monthly AS (
  SELECT strftime('%Y-%m', order_date) AS month,
         SUM(amount_kes)               AS revenue_kes
  FROM orders_clean
  WHERE status = 'completed'
    AND amount_outlier = 0
  GROUP BY strftime('%Y-%m', order_date)
)
SELECT month,
       revenue_kes,
       LAG(revenue_kes) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0 * (revenue_kes - LAG(revenue_kes) OVER (ORDER BY month))
             / LAG(revenue_kes) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month
"""))


amt = orders_clean.query("status == 'completed'")["amount_kes"].dropna()
print(amt.mean(), amt.median())                       # 14540.0, 2400.0
amt2 = amt[amt < 50000]
print(amt2.mean(), amt2.median())                     # 2364.3, 2375.0


from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import numpy as np

z, p = proportions_ztest([60, 50], [1000, 1000])
print(round(z, 2), round(p, 3))          # 0.98, 0.327

diff = 0.06 - 0.05
se = np.sqrt(0.05*0.95/1000 + 0.06*0.94/1000)
print(diff - 1.96*se, diff + 1.96*se)    # about -0.010, 0.030



import pandas as pd

df = pd.DataFrame({
    "channel": ["Organic", "Paid Social"],
    "sep_sessions": [40000, 10000], "sep_orders": [2400, 200],
    "oct_sessions": [38000, 30000], "oct_orders": [2318, 630],
})
df["sep_cr"] = df["sep_orders"] / df["sep_sessions"]
df["oct_cr"] = df["oct_orders"] / df["oct_sessions"]
df["sep_share"] = df["sep_sessions"] / df["sep_sessions"].sum()
df["oct_share"] = df["oct_sessions"] / df["oct_sessions"].sum()
print(df[["channel", "sep_cr", "oct_cr", "sep_share", "oct_share"]].round(4))

sep_total = df["sep_orders"].sum() / df["sep_sessions"].sum()
oct_total = df["oct_orders"].sum() / df["oct_sessions"].sum()
counterfactual = (df["sep_share"] * df["oct_cr"]).sum()   # Oct rates, Sep mix
print(round(sep_total, 4), round(oct_total, 4), round(counterfactual, 4))
# 0.052, 0.0434, 0.053